In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

import matplotlib.pyplot as plt

## Data Preprocessing

Read in process_uber_summary.parquet (Can be downloaded from https://gdo168.llnl.gov/data/ACME4/gold/process_uber_summary.parquet) Grab the one in the "gold" folder. It has the "red_team" column!

In [ ]:
orig_df = pd.read_parquet("process_uber_summary.parquet")

I have an easier time making sense of all the columns if I rename them:

In [ ]:
orig_df = orig_df.rename(columns={
    "filename": "process_filename",
    "file_id": "process_file_id",
    "file_md5": "process_file_md5",
    "num_file_md5": "process_num_file_md5",
    "file_sha2": "process_file_sha2",
    "num_file_sha2": "process_num_file_sha2",
    "first_seen": "process_first_seen",
    "last_seen": "process_last_seen",
    "process_started_seconds": "process_start_seconds",
    "process_started": "process_start",
    "process_term": "process_stop",
    "read_operation_count": "process_read_operation_count",
    "write_operation_count": "process_write_operation_count",
    "read_transfer_kilobytes": "process_read_transfer_kilobytes",
    "write_transfer_kilobytes": "process_write_transfer_kilobytes",
    "cpu_cycle_count": "process_cpu_cycle_count",
    "cpu_utilization": "process_cpu_utilization",
    "commit_charge": "process_commit_charge",
    "commit_peak": "process_commit_peak",
    "hard_fault_count": "process_hard_fault_count",
    "token_elevation_type": "process_token_elevation_type",
    "exit_code": "process_exit_code",
    "num_process_start": "process_num_process_start",
    "num_process_stop": "process_num_process_stop",
    "duration_seconds": "process_duration_seconds",
    "Read_Bytes": "file_read_bytes",
    "Read_Events": "file_read_events",
    "Write_Bytes": "file_write_bytes",
    "Write_Events": "file_write_events",
    "Close_Events": "file_close_events",
    "Create_Events": "file_create_events",
    "Delete_Events": "file_delete_events",
    "Rename_Events": "file_rename_events",
    "SetInfo_Events": "file_setInfo_events",
    "num_uniq_file_hash": "file_num_uniq_hashes",
    "num_null_filename": "file_num_null_filenames",
    "conn_id_count": "network_conn_id_count",
    "min_bytes": "network_min_bytes",
    "max_bytes": "network_max_bytes",
    "avg_bytes": "network_avg_bytes",
    "min_packets": "network_min_packets",
    "max_packets": "network_max_packets",
    "avg_packets": "network_avg_packets",
    "sq_size": "network_sq_size",
    "tcp_accept_count": "network_tcp_accept_count",
    "tcp_connect_count": "network_tcp_connect_count",
    "tcp_disconnect_count": "network_tcp_disconnect_count",
    "tcp_reconnect_count": "network_tcp_reconnect_count",
    "tcp_recv_count": "network_tcp_recv_count",
    "tcp_recv_size": "network_tcp_recv_size",
    "tcp_retransmit_count": "network_tcp_retransmit_count",
    "tcp_send_count": "network_tcp_send_count",
    "tcp_send_size": "network_tcp_send_size",
    "tcp_tcpcopy_count": "network_tcp_tcpcopy_count",
    "tcp_tcpcopy_size": "network_tcp_tcpcopy_size",
    "udp_recv_count": "network_udp_recv_count",
    "udp_recv_size": "network_udp_recv_size",
    "udp_send_count": "network_udp_send_count",
    "udp_send_size": "network_udp_send_size",
    "tcp_rs_total": "network_tcp_rs_total",
    "udp_rs_total": "network_udp_rs_total",
    "udp_send_vs_recv": "network_udp_send_vs_recv",
    "tcp_send_vs_recv": "network_tcp_send_vs_recv",
    "high_num_sigma_hits": "sigma_hits_num_high",
    "high_num_sigma_rows": "sigma_rows_num_high",
    "low_num_sigma_hits": "sigma_hits_num_low",
    "low_num_sigma_rows": "sigma_rows_num_low",
    "medium_num_sigma_hits": "sigma_hits_num_medium",
    "medium_num_sigma_rows": "sigma_rows_num_medium",
    "critical_num_sigma_hits": "sigma_hits_num_critical",
    "critical_num_sigma_rows": "sigma_rows_num_critical",
    "total_sigma_hits": "sigma_hits_total",
    "arch": "architecture",
    "net_total_events": "network_total_events",
    "net_total_size": "network_total_size",
    "net_num_raw_rows": "network_num_raw_rows",
    "net_recv_size": "network_recv_size",
    "net_send_size": "network_send_size",
    "net_rs_total": "network_rs_total",
    "net_send_vs_recv": "network_send_vs_recv",
    "net_first_seen": "network_first_seen",
    "net_last_seen": "network_last_seen",
    "lolc_class": "label_lolc_class",
    "bad_user": "label_bad_user",
    "red_team": "label_red_team",
})

And if I group them by type:

In [ ]:
new_column_order = [
    "pid_hash",
    "os_pid",
    "os_family",
    "os",
    "os_version",
    "architecture",
    "agent_id",
    "num_agent_id",
    "hostname",
    "process_name",
    "num_process_name",
    "args",
    "num_args",
    "user_name",
    "num_user_name",
    "parent_pid_hash",
    "num_parent_pid_hash",
    "parent_os_pid",
    "num_parent_os_pid",
    "process_path",
    "num_process_path",
    "process_filename",
    "process_file_id",
    "process_file_md5",
    "process_num_file_md5",
    "process_file_sha2",
    "process_num_file_sha2",
    "process_first_seen",
    "process_last_seen",
    "process_start_seconds",
    "process_start",
    "process_num_process_start",
    "process_stop_seconds",
    "process_stop",
    "process_num_process_stop",
    "process_duration_seconds",
    "process_cpu_cycle_count",
    "process_cpu_utilization",
    "process_commit_charge",
    "process_commit_peak",
    "process_read_operation_count",
    "process_write_operation_count",
    "process_read_transfer_kilobytes",
    "process_write_transfer_kilobytes",
    "process_hard_fault_count",
    "process_token_elevation_type",
    "process_exit_code",
    "reg_totals",
    "reg_reads",
    "reg_writes",
    "reg_createkeys",
    "reg_deletekeys",
    "reg_deletevalues",
    "reg_first_seen",
    "reg_last_seen",
    "file_close_events",
    "file_create_events",
    "file_delete_events",
    "file_rename_events",
    "file_setInfo_events",
    "file_read_bytes",
    "file_read_events",
    "file_write_bytes",
    "file_write_events",
    "file_num_raw_rows",
    "file_num_uniq_hashes",
    "file_num_null_filenames",
    "file_first_seen",
    "file_last_seen",
    "network_conn_id_count",
    "network_total_events",
    "network_total_size",
    "network_num_raw_rows",
    "network_tcp_accept_count",
    "network_tcp_connect_count",
    "network_tcp_disconnect_count",
    "network_tcp_reconnect_count",
    "network_tcp_recv_count",
    "network_tcp_recv_size",
    "network_tcp_retransmit_count",
    "network_tcp_send_count",
    "network_tcp_send_size",
    "network_tcp_tcpcopy_count",
    "network_tcp_tcpcopy_size",
    "network_udp_recv_count",
    "network_udp_recv_size",
    "network_udp_send_count",
    "network_udp_send_size",
    "network_recv_size",
    "network_send_size",
    "network_rs_total",
    "network_send_vs_recv",
    "network_tcp_rs_total",
    "network_tcp_send_vs_recv",
    "network_udp_rs_total",
    "network_udp_send_vs_recv",
    "network_min_bytes",
    "network_max_bytes",
    "network_avg_bytes",
    "network_min_packets",
    "network_max_packets",
    "network_avg_packets",
    "network_sq_size",
    "network_first_seen",
    "network_last_seen",
    "dlls",
    "dll_num_uniq_files",
    "dll_first_seen",
    "dll_last_seen",
    "sigma_hits_num_high",
    "sigma_rows_num_high",
    "sigma_hits_num_low",
    "sigma_rows_num_low",
    "sigma_hits_num_medium",
    "sigma_rows_num_medium",
    "sigma_hits_num_critical",
    "sigma_rows_num_critical",
    "sigma_hits_total",
    "lolbas_privs",
    "lolbas_cats",
    "lolbas_mitre",
    "lolbas_num_rows",
    "mitre_analytic_ids",
    "mitre_information_domains",
    "mitre_subtypes",
    "mitre_analytic_types",
    "mitre_num_rows",
    "label_lolc_class",
    "label_num_sources",
    "label_num_uniq_annotations",
    "label_sources",
    "label_annonations",
    "label_num_hits",
    "label_bad_user",
    "label_red_team"
]
orig_df = orig_df[new_column_order]

Removing the top three most common processes cuts out 93% of our overall data while removing only a small fraction of the processes marked as red team activity:

In [ ]:
orig_df["process_name"].value_counts()[0:20]

In [ ]:
df = orig_df[~orig_df["process_name"].isin(["conhost.exe","wmic.exe","mergehelper.exe"])].reset_index().drop(columns=["index"])

In [ ]:
print(f"Removing conhost.exe, wmic.exe, and mergehelper.exe takes the dataset down from {len(orig_df)} to {len(df)} instances")
new_num_red = df["label_red_team"].sum()
orig_num_red = orig_df["label_red_team"].sum()
print(f"This means we keep {new_num_red} of our {orig_num_red} red team instances")

## IRONMINER

Note that IRONMINER isn't pip installable yet. (We're still working on getting permission.) Please see Kristine for the code.

In [ ]:
import ironminer as ir

## Supervised Learning

Split the data into training and testing sets since we're going to be doing supervised learning experiments

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df["label_red_team"], random_state=42)

Use the training set to create a model for each column and do probability density estimation on all values in both the training and testing sets.

IRONMINER uses categorical model for strings and Gaussian for numerical values as a default. Probability density measures for each value in the dataset can be found in anomscores.

In [ ]:
anomdetect = ir.Anomdetect([df_train, df_test], model_idxs=[0])
anomdetect.process_datasets()
anomscores = anomdetect.anomscores

In [ ]:
anomscores[new_column_order]

Remove metadata columns, timing columns, and columns with hand-applied labels:

In [ ]:
metadata_columns = new_column_order[0:35]
time_columns = [c for c in df.columns if "start" in c or "stop" in c or "seen" in c]
label_columns = [c for c in df.columns if c.startswith("label_")]
drop_columns = metadata_columns + time_columns + label_columns

And train a random forest to predict the red team activity:

In [ ]:
X_train = anomscores[0:len(df_train)].drop(columns=drop_columns)
y_train = df_train["label_red_team"]
X_test = anomscores[len(df_train):].drop(columns=drop_columns)
y_test = df_test["label_red_team"]

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(confusion_matrix(y_test, y_pred))

In [ ]:
feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print(feature_importance[:20])

In [ ]:
df_pos = df[df["label_red_team"]==1]
df_neg = df[df["label_red_team"]==0]
fig = ir.get_plot([df_neg, df_pos], "process_name", 
                legend_names=["Background", "Red Team"], legend_colors=["blue", "red"],
                title="process_name")
display(fig)
fig = ir.get_plot([df_neg, df_pos], "process_token_elevation_type", 
                legend_names=["Background", "Red Team"], legend_colors=["blue", "red"],
                title="process_token_elevation_type", plot_type="unique_values")
display(fig)

Test the effectiveness of different feature types for predicting the red team activity. (Note that these won't necessarily be the columns that will predict ALL malicious activity, but it will give an idea of which feature types are good for fingerprinting characteristic behaviors of users, etc.)

In [ ]:
feature_types = ["process", "file", "network", "dll", "reg", "sigma", "lolbas", "mitre"]

for feature_type in feature_types:
    target_columns = [c for c in df_train.columns if c.startswith(feature_type) and c not in drop_columns]
    X_train = anomscores[0:len(df_train)][target_columns]
    y_train = df_train["label_red_team"]
    X_test = anomscores[len(df_train):][target_columns]
    y_test = df_test["label_red_team"]

    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print(feature_type)
    print(cm)

    '''
    y_prob = rf.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{feature_type} features")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()
    '''

## Anomaly Detection

For each column in anomscores, IRONMINER can sort likelihoods from least to most likely and calculate metrics on how closely the red team activity lands to the top. The "false alarms before detection" (FABD) metric reports how many background instances are less likely than the least likely red team instance. The "area under the curve" (AUC) metric reports relative likelihoods of background and red team instances across the entire dataset. The "precision at N" metric reports the percentage of red team instances in the top N least likely values (default N is 100).

In [ ]:
df_pos = df[df["label_red_team"]==1]
df_neg = df[df["label_red_team"]==0]

anomdetect2 = ir.Anomdetect([df_neg.drop(columns=drop_columns),df_pos.drop(columns=drop_columns)], model_idxs=[0])
anomdetect2.process_datasets()
anomscores2 = anomdetect2.anomscores
compare_datasets = anomdetect2.compare_datasets(N=100)

In [ ]:
compare_datasets.fabd_sort

In [ ]:
compare_datasets.auc_sort

In [ ]:
compare_datasets.precisionAtN_sort

In [ ]:
compare_datasets.topN["sigma_hits_total"][0:100]